# Notebook 03 — LSTM Training
Trains the main LSTM classifier on CIC-DDoS2019 with hyperparameter tuning.

In [ ]:
import subprocess, os
from pathlib import Path

REPO_URL  = 'https://github.com/calvinkatoroy/tugas-akhir-ai-kel-08.git'
REPO_NAME = 'tugas-akhir-ai-kel-08'

cwd = Path.cwd()

if (cwd / '../src').resolve().exists():
    result = subprocess.run(['git', '-C', str((cwd / '..').resolve()), 'pull'],
                            capture_output=True, text=True)
    print(result.stdout.strip() or 'Already up to date.')

elif (cwd / 'src').exists():
    result = subprocess.run(['git', 'pull'], capture_output=True, text=True)
    print(result.stdout.strip() or 'Already up to date.')
    os.chdir('notebooks')

else:
    repo_path = cwd / REPO_NAME
    if not repo_path.exists():
        print(f'Cloning {REPO_URL} ...')
        subprocess.run(['git', 'clone', REPO_URL], check=True)
    else:
        print('Repo found, pulling latest...')
        subprocess.run(['git', '-C', str(repo_path), 'pull'], capture_output=True)
    os.chdir(repo_path / 'notebooks')

print(f'Working dir: {Path.cwd()}')

In [ ]:
import sys
sys.path.insert(0, '..')

import random
import copy
import numpy as np
import torch
import yaml
from pathlib import Path

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

In [ ]:
with open('../config.yaml') as f:
    cfg = yaml.safe_load(f)

splits_dir = Path('../' + cfg['data']['splits_path'])

X_train = np.load(splits_dir / 'X_train_seq.npy')
y_train = np.load(splits_dir / 'y_train_seq.npy')
X_val   = np.load(splits_dir / 'X_val_seq.npy')
y_val   = np.load(splits_dir / 'y_val_seq.npy')
X_test  = np.load(splits_dir / 'X_test_seq.npy')
y_test  = np.load(splits_dir / 'y_test_seq.npy')

n_features = X_train.shape[2]
seq_len    = X_train.shape[1]
print(f'Train: {X_train.shape}  Val: {X_val.shape}  Test: {X_test.shape}')
print(f'n_features={n_features}, seq_len={seq_len}')

## Baseline run (config defaults)

In [ ]:
from src.models.lstm_model import build_lstm
from src.train import train_model
from src.utils import log_experiment

model = build_lstm(cfg, n_features)
print(model)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Trainable parameters: {total_params:,}')

In [ ]:
history_baseline, ckpt_baseline = train_model(
    model, X_train, y_train, X_val, y_val,
    cfg=cfg, model_key='lstm',
    checkpoint_dir='../results/checkpoints',
)

log_experiment({
    'exp_id': 'lstm_01_baseline',
    'model': 'LSTM',
    'hidden_size': cfg['lstm']['hidden_size'],
    'num_layers': cfg['lstm']['num_layers'],
    'dropout': cfg['lstm']['dropout'],
    'seq_len': cfg['lstm']['seq_len'],
    'lr': cfg['lstm']['learning_rate'],
    'batch_size': cfg['lstm']['batch_size'],
    'best_val_f1': round(max(history_baseline['val_f1']), 4),
    'notes': 'baseline — config defaults',
}, csv_path='../results/metrics_summary.csv')

## Hyperparameter Experiments
Document at least 5 runs. Change ONE hyperparam at a time.

In [ ]:
# Helper: run one experiment and log it
def run_lstm_experiment(exp_id, overrides, notes=''):
    """overrides: dict of cfg['lstm'] keys to override."""
    exp_cfg = copy.deepcopy(cfg)
    exp_cfg['lstm'].update(overrides)

    random.seed(SEED); np.random.seed(SEED)
    torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

    m = build_lstm(exp_cfg, n_features)
    hist, ckpt = train_model(
        m, X_train, y_train, X_val, y_val,
        cfg=exp_cfg, model_key='lstm',
        checkpoint_dir='../results/checkpoints',
    )
    best_f1 = round(max(hist['val_f1']), 4)
    log_experiment({
        'exp_id': exp_id,
        'model': 'LSTM',
        'hidden_size': exp_cfg['lstm']['hidden_size'],
        'num_layers': exp_cfg['lstm']['num_layers'],
        'dropout': exp_cfg['lstm']['dropout'],
        'seq_len': exp_cfg['lstm']['seq_len'],
        'lr': exp_cfg['lstm']['learning_rate'],
        'batch_size': exp_cfg['lstm']['batch_size'],
        'best_val_f1': best_f1,
        'notes': notes,
    }, csv_path='../results/metrics_summary.csv')
    print(f'[{exp_id}] best_val_f1={best_f1}')
    return hist, ckpt, best_f1

In [ ]:
# Exp 02 — smaller hidden size
hist_02, ckpt_02, f1_02 = run_lstm_experiment(
    'lstm_02_hidden64', {'hidden_size': 64}, 'hidden_size=64')

In [ ]:
# Exp 03 — larger hidden size
hist_03, ckpt_03, f1_03 = run_lstm_experiment(
    'lstm_03_hidden256', {'hidden_size': 256}, 'hidden_size=256')

In [ ]:
# Exp 04 — 3 layers
hist_04, ckpt_04, f1_04 = run_lstm_experiment(
    'lstm_04_layers3', {'num_layers': 3}, 'num_layers=3')

In [ ]:
# Exp 05 — lower learning rate
hist_05, ckpt_05, f1_05 = run_lstm_experiment(
    'lstm_05_lr1e4', {'learning_rate': 1e-4}, 'lr=1e-4')

In [ ]:
# Exp 06 — shorter sequence window
# NOTE: requires rebuilding sequences at seq_len=5
# Load the flat splits and re-window
from src.preprocessing import make_sequences
import joblib

X_tr_flat = np.load(splits_dir / 'X_train.npy')
y_tr_flat = np.load(splits_dir / 'y_train.npy')
X_vl_flat = np.load(splits_dir / 'X_val.npy')
y_vl_flat = np.load(splits_dir / 'y_val.npy')

X_tr5, y_tr5 = make_sequences(X_tr_flat, y_tr_flat, seq_len=5)
X_vl5, y_vl5 = make_sequences(X_vl_flat, y_vl_flat, seq_len=5)

exp_cfg6 = copy.deepcopy(cfg)
exp_cfg6['lstm']['seq_len'] = 5
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
m6 = build_lstm(exp_cfg6, n_features)
hist_06, ckpt_06 = train_model(
    m6, X_tr5, y_tr5, X_vl5, y_vl5,
    cfg=exp_cfg6, model_key='lstm',
    checkpoint_dir='../results/checkpoints',
)
f1_06 = round(max(hist_06['val_f1']), 4)
log_experiment({
    'exp_id': 'lstm_06_seqlen5',
    'model': 'LSTM',
    'hidden_size': 128, 'num_layers': 2, 'dropout': 0.3,
    'seq_len': 5, 'lr': 0.001, 'batch_size': 64,
    'best_val_f1': f1_06, 'notes': 'seq_len=5',
}, csv_path='../results/metrics_summary.csv')
print(f'[lstm_06_seqlen5] best_val_f1={f1_06}')

## Final Evaluation on Test Set

In [ ]:
# Identify best experiment by val F1
import pandas as pd

results = pd.read_csv('../results/metrics_summary.csv')
lstm_results = results[results['model'] == 'LSTM'].copy()
print(lstm_results[['exp_id', 'hidden_size', 'num_layers', 'dropout',
                     'seq_len', 'lr', 'best_val_f1', 'notes']])

best_row = lstm_results.loc[lstm_results['best_val_f1'].idxmax()]
print(f'\nBest experiment: {best_row["exp_id"]}  val_F1={best_row["best_val_f1"]}')

In [ ]:
# Load best checkpoint and evaluate on test set
from src.models.lstm_model import build_lstm
from src.evaluate import evaluate_dl_model
from src.utils import plot_loss_curves

best_model = build_lstm(cfg, n_features).to(device)
best_model.load_state_dict(torch.load('../results/checkpoints/best_lstm.pt', map_location=device))

y_pred_lstm, y_prob_lstm = evaluate_dl_model(
    best_model, X_test, y_test,
    model_name='LSTM',
    history=history_baseline,
    figures_dir='../results/figures',
    csv_path='../results/metrics_summary.csv',
)

# Save probs for ROC comparison in notebook 06
np.save('../results/lstm_y_prob.npy', y_prob_lstm)
np.save('../results/y_test_seq.npy', y_test)

In [ ]:
# Plot loss curve for best run
plot_loss_curves(
    history_baseline['train_loss'], history_baseline['val_loss'],
    title='LSTM Loss Curves (baseline)',
    save_path='../results/figures/loss_lstm.png',
)